# WinCLIP MVTec Experiment

MVTec 데이터셋에서 `models.winclip.WinCLIP`을 PatchCore와 같은 `fit / predict / predict_batch` 인터페이스로 실행합니다.

In [4]:
import os
import sys

# Notebook을 MVTec 폴더 안에서 열어도 repo root 기준으로 실행되게 맞춥니다.
if os.path.basename(os.getcwd()) == "MVTec":
    os.chdir("..")

print(os.getcwd())

sys.path.append("..")

/Users/taehayeong/Desktop/lab


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "MVTec":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import numpy as np

import config_mv
from datasets.mvtec import MyData
from models.winclip import WinCLIP
from utils.metrics import get_image_auc
from utils.visualization import show_result

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(device)

mps


In [3]:
train_data = MyData(
    config_mv.CLASS_NAME,
    phase="train",
    batch_size=config_mv.BATCH_SIZE,
    shuffle=False,
    limit=config_mv.TRAIN_LIMIT,
)

test_data = MyData(
    config_mv.CLASS_NAME,
    phase="test",
    batch_size=config_mv.BATCH_SIZE,
    shuffle=False,
    limit=config_mv.TEST_LIMIT,
    limit_per_class=config_mv.TEST_LIMIT_PER_CLASS,
)

print("class:", config_mv.CLASS_NAME)
print("train:", len(train_data), "test:", len(test_data))

FileNotFoundError: [Errno 2] No such file or directory: 'data/MVTec/cable/train'

In [ ]:
winclip = WinCLIP(
    category=config_mv.CLASS_NAME,
    device=device,
    backbone="ViT-B-16-plus-240",
    pretrained_dataset="laion400m_e32",
    out_size_h=256,
    out_size_w=256,
    img_resize=256,
    img_cropsize=240,
    use_visual_gallery=True,
)

winclip.fit(train_data)

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
scores = []
heatmaps = []
imgs = []

for img, label in test_data:
    score, heatmap = winclip.predict(img)
    scores.append(score.item())
    heatmaps.append(heatmap)
    imgs.append(img)

print("scores:", np.round(scores, 4).tolist())

In [ ]:
image_auc = get_image_auc(test_data, winclip)
print("image AUROC:", image_auc)

In [ ]:
show_result(test_data, scores, heatmaps, k=4)